# DLT Workshop

**Table of Content**

- [Workshop](##Workshop)
- [Homework](##Homework)

<hr>

## Workshop

### Intro

The data load tool (dlt) is an open-source Python library designed to simplify data ingestion and loading. 

Common use-cases are:

- Extracting data from APIs, databases, files or SaaS tools
- load data into warehouses and databases

DlT is therefore comparable to a lightweit ELT (Extract-Load-Transform) framework

### Installation

The installation requires to both set up a MCP Server and the actual installation of dlt

After creating an MCP Server dlt is installed. I used conda for a clean installation in an own environment. 

In [ ]:
conda create -n dlt_env python=3.11
conda activate dlt_env
pip install "dlt[workspace]"

### Setup

After installation the project of the workshop was setup as followed

In [ ]:
dlt init dlthub:taxi_pipeline duckdb

### Creating Pipeline

The pipeline was created with Microsoft Copilot with following prompt:


```

You are helping me implement a working dlt pipeline.

Goal:
Create a complete dlt pipeline that loads NYC taxi trip data from a REST API into DuckDB so that analytical SQL queries can be run afterwards.

Project setup already exists:
- taxipipeline.py (this file)
- config.toml
- requirements.txt
- duckdb destination configured

Requirements:

1. Build a dlt REST API source using @dlt.source and @dlt.resource.

2. API details:
- Base URL:
  https://us-central1-dlthub-analytics.cloudfunctions.net/data_engineering_zoomcamp_api
- Response format: JSON list of records
- Pagination:
  - Page-based pagination
  - Each page returns up to 1000 records
  - Use query parameter `page`
  - Continue requesting pages until an empty list is returned

3. Implementation details:
- Create a source named `taxi_source`
- Create a resource named `taxi_trips`
- Yield records page by page
- Do NOT load all data into memory at once
- Use requests library
- Handle pagination in a loop

4. Pipeline:
- Pipeline name: "taxi_pipeline"
- Destination: duckdb
- Dataset name: "taxi_data"

5. Execution:
- Include a main block so running:
      python taxipipeline.py
  executes the pipeline.

6. Output:
- Print load information after completion.

7. Code quality:
- Minimal but production-style code
- Clear structure
- No placeholders
- Fully runnable without manual edits

Important:
The resulting DuckDB table must allow answering these SQL questions:

- dataset start date and end date
- proportion of trips paid with credit card
- total sum of tips

Assume fields like pickup_datetime, payment_type, and tip_amount exist in the API response.

Generate the full working file.



```

## Homework

After creating the pipeline I responded to the Homework Questions

### Question 1


Question 1: What is the start date and end date of the dataset?


- <span style = "background-color: red; color: white; opacity: .7">2009-01-01 to 2009-01-31</span>

-  <span style = "background-color: green; color: white; opacity: .7">2009-06-01 to 2009-07-01</span>


-  <span style = "background-color: red; color: white; opacity: .7">2024-01-01 to 2024-02-01</span>


-  <span style = "background-color: red; color: white; opacity: .7">2024-06-01 to 2024-07-01</span>

**Query**

In [ ]:
SELECT
    MIN(trip_pickup_date_time) AS dataset_start_date,
    MAX(trip_dropoff_date_time) AS dataset_end_date
FROM taxi_data.taxi_trips;

### Question 2


Question 2:  What proportion of trips are paid with credit card?


- <span style = "background-color: red; color: white; opacity: .7">16.66%</span>

-  <span style = "background-color: green; color: white; opacity: .7">26.66%</span>


-  <span style = "background-color: red; color: white; opacity: .7">36.66%</span>


-  <span style = "background-color: red; color: white; opacity: .7">46.66%</span>

**Query**

In [ ]:
SELECT
    AVG(
        CASE
            WHEN lower(payment_type) IN ('credit', 'credit card', 'card') THEN 1.0
            ELSE 0.0
        END
    ) AS credit_card_proportion
FROM taxi_data.taxi_trips;

### Question 3


Question 3: What is the total amount of money generated in tips?


- <span style = "background-color: red; color: white; opacity: .7">$4,063.41</span>

-  <span style = "background-color: green; color: white; opacity: .7">$6,063.41</span>


-  <span style = "background-color: red; color: white; opacity: .7">$8,063.41</span>


-  <span style = "background-color: red; color: white; opacity: .7">$10,063.41</span>

**Query**

In [ ]:
SELECT
    SUM(COALESCE(tip_amt, 0)) AS total_tip_amount
FROM taxi_data.taxi_trips;